# Week 4 — Ontology

This week formalizes something Weeks 1 and 3 only handled empirically.

- **Week 1 Day 1** found that a weak embedding model ranked a "customer support ticket" (a complaint) *closer* to "customer placed an order" (an order) than to its true pair — a misranking that was fixed only by switching embedding models. Nothing *prevented* that confusion; we just got lucky with a better model.
- **Week 3** kept `Order` and `InventoryReplenishment` separate by giving them no edge in the Neo4j graph. That worked too — but only because nobody happened to draw that edge. Neo4j has no concept that these are fundamentally different *kinds* of thing; `:Order` and `:InventoryReplenishment` are just string labels to it.

This week's question: can we state, as a **formal rule a machine can check**, that two classes of thing can never overlap — so that the Week 1 mistake becomes a detectable contradiction instead of a silent ranking error?

**Setup** (run once):
```
pip install rdflib owlrl
```


In [ ]:
!pip install rdflib owlrl -q

## Day 1 — Declaring formal disjointness

We build a tiny ontology with `rdflib`:
- Two classes, `Order` and `Complaint`.
- An `owl:disjointWith` axiom between them — the formal claim that **no individual can ever be both at once**.
- One real instance of each, mirroring Week 1's actual `[0]` (an order) and `[7]` (a complaint).

Run this first and just confirm it builds — you should see 7 triples (2 class declarations, 1 disjointness axiom, 2 `rdf:type` assertions, 2 labels).

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
from rdflib.namespace import XSD

ns = Namespace("http://example.org/biz#")
g = Graph()
g.bind("biz", ns)

# Declare two classes
g.add((ns.Order, RDF.type, OWL.Class))
g.add((ns.Complaint, RDF.type, OWL.Class))

# The key formal move: these classes can NEVER share an individual
g.add((ns.Order, OWL.disjointWith, ns.Complaint))

# Create one real instance of each — mirroring Week 1's actual [0] and [7]
g.add((ns.order_0, RDF.type, ns.Order))
g.add((ns.order_0, RDFS.label, Literal("Customer placed an order for laptops")))

g.add((ns.complaint_7, RDF.type, ns.Complaint))
g.add((ns.complaint_7, RDFS.label, Literal("Customer support ticket opened for defective item")))

print(f"Triples before reasoning: {len(g)}")
for s, p, o in g:
    print(s, p, o)

## Day 1 (continued) — deliberately committing the Week 1 mistake

Now we deliberately do what a weak embedding ranking effectively encouraged: assert that `complaint_7` is **also** an `Order`. Then we run an actual OWL reasoner (`owlrl`) over the graph and see what it does with a graph that now contains a logical contradiction.

Run this and report back exactly what you see — does it raise an error, print nothing, or do something else? That result decides what Day 2 builds on.

In [ ]:
import owlrl

# Deliberately misclassify complaint_7 as ALSO an Order —
# this is precisely the kind of error a weak embedding model's
# ranking could lead someone to encode by hand
g.add((ns.complaint_7, RDF.type, ns.Order))

print(f"Triples after the bad assertion: {len(g)}")

# Run OWL-RL inference/consistency checking over the graph
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

print(f"Triples after reasoning: {len(g)}")